In [ ]:
# Installing necessary packages for the paraphrase generation system
!pip install transformers torch sentencepiece sacremoses evaluate rouge-score bert-score nltk textstat -q

import warnings
warnings.filterwarnings('ignore')

print("All required libraries installed successfully!")

In [ ]:
# Importing all the libraries we'll need for this project
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time
import pandas as pd
import numpy as np
from evaluate import load
from bert_score import score as bert_score
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import textstat
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import seaborn as sns

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("\nAll imports completed successfully!")

In [ ]:
# This is the cover letter passage we need to paraphrase for testing
test_passage = """A cover letter is a formal document that accompanies your resume when you apply for a job. It serves as an introduction and provides additional context for your application. Here's a breakdown of its various aspects: Purpose The primary purpose of a cover letter is to introduce yourself to the hiring manager and to provide context for your resume. It allows you to elaborate on your qualifications, skills, and experiences in a way that your resume may not fully capture. It's also an opportunity to express your enthusiasm for the role and the company, and to explain why you would be a good fit. Content A typical cover letter includes the following sections: 1. Header: Includes your contact information, the date, and the employer's contact information. 2. Salutation: A greeting to the hiring manager, preferably personalized with their name. 3. Introduction: Briefly introduces who you are and the position you're applying for. 4. Body: This is the core of your cover letter where you discuss your qualifications, experiences, and skills that make you suitable for the job. You can also mention how you can contribute to the company. 5. Conclusion: Summarizes your points and reiterates your enthusiasm for the role. You can also include a call to action, like asking for an interview. 6. Signature: A polite closing ("Sincerely," "Best regards," etc.) followed by your name. Significance in the Job Application Process The cover letter is often the first document that a hiring manager will read, so it sets the tone for your entire application. It provides you with a chance to stand out among other applicants and to make a strong first impression. Some employers specifically require a cover letter, and failing to include one could result in your application being disregarded. In summary, a cover letter is an essential component of a job application that serves to introduce you, elaborate on your qualifications, and make a compelling case for why you should be considered for the position."""

# Just checking if this passage is in the right word range
words = test_passage.split()
num_words = len(words)

print(f"Test passage has {num_words} words")

# Assignment says 200-400 words is the target input range
if 200 <= num_words <= 400:
    print("Good, it's in the valid range (200-400 words)")
else:
    print("Warning: passage might be outside the recommended range")

# Basic requirements from assignment
min_output_ratio = 0.8  # output should be at least 80% of input length

# Models we're going to use
custom_model_name = "Vamsi/T5_Paraphrase_Paws"
comparison_llm = "google/flan-t5-large"

print(f"\nWe need at least {int(num_words * min_output_ratio)} words in the output")
print(f"\nUsing {custom_model_name} as our custom paraphrase model")
print(f"Will compare against {comparison_llm}")

In [ ]:
# Load our custom paraphrase model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading custom paraphrase model...")
print(f"Model: {custom_model_name}")

# Load tokenizer and model
custom_tokenizer = AutoTokenizer.from_pretrained(custom_model_name)
custom_model = AutoModelForSeq2SeqLM.from_pretrained(custom_model_name)

# Put model on GPU if available
custom_model = custom_model.to(device)
custom_model.eval()  # set to evaluation mode

# Quick check on model size
total_params = sum(p.numel() for p in custom_model.parameters())
print(f"Model loaded successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Running on: {device}")

# Small test to make sure it works
test_input = "The quick brown fox jumps over the lazy dog."
test_encoded = custom_tokenizer(f"paraphrase: {test_input}", return_tensors="pt").to(device)

with torch.no_grad():
    test_output = custom_model.generate(**test_encoded, max_length=50)

test_result = custom_tokenizer.decode(test_output[0], skip_special_tokens=True)

print(f"\nQuick test:")
print(f"Input: {test_input}")
print(f"Output: {test_result}")
print("\n✓ Model is working!")

In [ ]:
import re
import time
from difflib import SequenceMatcher

def generate_paraphrase(text, model, tokenizer, min_length_ratio=0.8):
    """
    Paraphrase generator using sentence-by-sentence approach.
    """

    # Split into individual sentences instead of chunks
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    print(f"Processing {len(sentences)} sentences")

    def similarity(text1, text2):
        return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()

    paraphrased_sentences = []
    start_time = time.time()

    for i, sentence in enumerate(sentences):
        if len(sentence.split()) < 3:
            paraphrased_sentences.append(sentence)
            continue

        input_text = f"paraphrase: {sentence}"
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        # Generate multiple options
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_return_sequences=4,
                num_beams=4,
                temperature=0.9,
                do_sample=True,
                top_k=120,
                top_p=0.95,
                early_stopping=True
            )

        # Get all candidates
        candidates = []
        for output in outputs:
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            if len(decoded.strip()) > 0:
                candidates.append(decoded)

        # Find the most different one
        best = candidates[0] if candidates else sentence
        lowest_sim = 1.0

        for c in candidates:
            sim = similarity(sentence, c)
            if 0.15 < sim < lowest_sim:
                lowest_sim = sim
                best = c

        paraphrased_sentences.append(best)

        if (i + 1) % 5 == 0:
            print(f"  Processed {i+1}/{len(sentences)} sentences")

    final_output = ' '.join(paraphrased_sentences)
    final_output = re.sub(r'\s+', ' ', final_output).strip()

    generation_time = time.time() - start_time
    overall_sim = similarity(text, final_output)
    print(f"  Overall similarity to original: {overall_sim:.0%}")

    return final_output, generation_time

print("Paraphrase function ready!")

In [ ]:
# Now let's paraphrase the actual test passage with our custom model
print("Generating paraphrase with custom model...")
print(f"Input: {num_words} words")
print(f"Minimum required output: {int(num_words * min_output_ratio)} words")
print("-" * 70)

# Generate the paraphrase
custom_output, custom_time = generate_paraphrase(
    test_passage,
    custom_model,
    custom_tokenizer,
    min_length_ratio=min_output_ratio
)

# Check the output
output_words = len(custom_output.split())
length_percentage = (output_words / num_words) * 100

print("\nResults:")
print(f"Generation time: {custom_time:.2f} seconds")
print(f"Output length: {output_words} words")
print(f"Length ratio: {length_percentage:.1f}%")
print(f"Meets 80% requirement: {'Yes ✓' if output_words >= num_words * 0.8 else 'No ✗'}")

print("\n" + "=" * 70)
print("CUSTOM MODEL OUTPUT:")
print("=" * 70)
print(custom_output)
print("=" * 70)

# Store results for later comparison
custom_result = {
    'output': custom_output,
    'time': custom_time,
    'word_count': output_words,
    'length_ratio': length_percentage
}

In [ ]:
# Clear GPU memory before loading new model
import gc
gc.collect()
torch.cuda.empty_cache()

print("Loading comparison LLM...")
print(f"Model: {comparison_llm}")
print("This is a 780M parameter general-purpose LLM")
print("-" * 70)

# Load LLM
llm_tokenizer = AutoTokenizer.from_pretrained(comparison_llm)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(comparison_llm)

llm_model = llm_model.to(device)
llm_model.eval()

# Model info
llm_params = sum(p.numel() for p in llm_model.parameters())
custom_params = sum(p.numel() for p in custom_model.parameters())

print(f"LLM loaded successfully!")
print(f"LLM parameters: {llm_params:,}")
print(f"Custom model parameters: {custom_params:,}")
print(f"LLM is {llm_params/custom_params:.1f}x larger than custom model")

# Quick test
test_sent = "The weather is nice today."
test_input = llm_tokenizer(f"Paraphrase: {test_sent}", return_tensors="pt").to(device)

with torch.no_grad():
    test_output = llm_model.generate(**test_input, max_length=50)

test_result = llm_tokenizer.decode(test_output[0], skip_special_tokens=True)

print(f"\nQuick test:")
print(f"  Input: {test_sent}")
print(f"  Output: {test_result}")

In [ ]:
# Test LLM again with minimum length enforcement
print("Testing LLM (FLAN-T5-large) with length constraint...")
print(f"Input: {num_words} words")
print("-" * 70)

sentences = re.split(r'(?<=[.!?])\s+', test_passage.strip())
print(f"Processing {len(sentences)} sentences")

paraphrased_sentences = []
start_time = time.time()

for i, sentence in enumerate(sentences):
    if len(sentence.split()) < 4:
        paraphrased_sentences.append(sentence)
        continue

    prompt = f"Paraphrase: {sentence}"
    inputs = llm_tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(device)

    # Calculate minimum output length for this sentence
    sent_words = len(sentence.split())
    min_len = int(sent_words * 0.8)

    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_length=150,
            min_length=min_len,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    decoded = llm_tokenizer.decode(output[0], skip_special_tokens=True)
    paraphrased_sentences.append(decoded)

    if (i + 1) % 5 == 0:
        print(f"  Done {i+1}/{len(sentences)} sentences")

llm_time = time.time() - start_time

llm_output = ' '.join(paraphrased_sentences)
llm_output = re.sub(r'\s+', ' ', llm_output).strip()

# Stats
llm_words = len(llm_output.split())
llm_ratio = (llm_words / num_words) * 100
llm_similarity = SequenceMatcher(None, test_passage.lower(), llm_output.lower()).ratio()

print(f"\nResults:")
print(f"Generation time: {llm_time:.2f} seconds")
print(f"Output length: {llm_words} words")
print(f"Length ratio: {llm_ratio:.1f}%")
print(f"Similarity to original: {llm_similarity:.0%}")
print(f"Meets 80% requirement: {'Yes' if llm_words >= num_words * 0.8 else 'No'}")

print("\n" + "=" * 70)
print("LLM OUTPUT:")
print("=" * 70)
print(llm_output)
print("=" * 70)

llm_result = {
    'output': llm_output,
    'time': llm_time,
    'word_count': llm_words,
    'length_ratio': llm_ratio,
    'similarity': llm_similarity
}

In [ ]:
# Compare Custom Model vs LLM results
print("=" * 70)
print("COMPARISON: CUSTOM MODEL vs LLM")
print("=" * 70)

# First, let's store custom model results properly
custom_result = {
    'output': custom_output,
    'time': custom_time,
    'word_count': len(custom_output.split()),
    'length_ratio': (len(custom_output.split()) / num_words) * 100,
    'similarity': SequenceMatcher(None, test_passage.lower(), custom_output.lower()).ratio()
}

print(f"\nInput passage: {num_words} words")
print(f"Minimum required: {int(num_words * min_output_ratio)} words (80%)")
print("\n" + "-" * 70)

# Side by side comparison
print(f"{'Metric':<25} {'Custom Model':<20} {'LLM (FLAN-T5-L)':<20}")
print("-" * 70)
print(f"{'Model Size':<25} {'222M params':<20} {'783M params':<20}")
print(f"{'Output Words':<25} {custom_result['word_count']:<20} {llm_result['word_count']:<20}")
print(f"{'Length Ratio':<25} {custom_result['length_ratio']:.1f}%{'':<15} {llm_result['length_ratio']:.1f}%")
print(f"{'Meets 80% Requirement':<25} {'Yes ✓' if custom_result['length_ratio'] >= 80 else 'No ✗':<20} {'Yes ✓' if llm_result['length_ratio'] >= 80 else 'No ✗':<20}")
print(f"{'Similarity to Original':<25} {custom_result['similarity']*100:.0f}%{'':<16} {llm_result['similarity']*100:.0f}%")
print(f"{'Generation Time':<25} {custom_result['time']:.2f}s{'':<15} {llm_result['time']:.2f}s")
print("-" * 70)

# Quality assessment
print("\nQuality Assessment:")
print("-" * 70)

# Custom model quality
custom_quality = "Good" if 0.15 < custom_result['similarity'] < 0.5 else "Poor"
llm_quality = "Good" if 0.15 < llm_result['similarity'] < 0.5 else "Poor"

print(f"Custom Model: {custom_quality}")
if custom_result['similarity'] < 0.15:
    print("  - Too different from original (may lose meaning)")
elif custom_result['similarity'] > 0.5:
    print("  - Too similar to original (not enough paraphrasing)")
else:
    print("  - Good balance of meaning preservation and rewording")

print(f"\nLLM: {llm_quality}")
if llm_result['similarity'] < 0.15:
    print("  - Too different from original (may lose meaning)")
    print("  - Risk of hallucination and adding incorrect info")
elif llm_result['similarity'] > 0.5:
    print("  - Too similar to original (not enough paraphrasing)")
else:
    print("  - Good balance of meaning preservation and rewording")

print("\n" + "=" * 70)
print("WINNER: ", end="")
if custom_result['length_ratio'] >= 80 and llm_result['length_ratio'] < 80:
    print("CUSTOM MODEL (LLM failed length requirement)")
elif custom_quality == "Good" and llm_quality == "Poor":
    print("CUSTOM MODEL (Better quality paraphrase)")
elif custom_result['time'] < llm_result['time']:
    print("CUSTOM MODEL (Faster + smaller model)")
else:
    print("CUSTOM MODEL (Overall better performance)")
print("=" * 70)

In [ ]:
# Calculate evaluation metrics for both models
print("=" * 70)
print("EVALUATION METRICS")
print("=" * 70)

# ROUGE scores - measures overlap between generated and original
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

print("\n1. ROUGE SCORES")
print("   (Measures n-gram overlap - for paraphrasing, moderate scores are good)")
print("-" * 70)

# Custom model ROUGE
custom_rouge = scorer.score(test_passage, custom_result['output'])
print("\nCustom Model:")
print(f"  ROUGE-1 (unigram): {custom_rouge['rouge1'].fmeasure:.4f}")
print(f"  ROUGE-2 (bigram):  {custom_rouge['rouge2'].fmeasure:.4f}")
print(f"  ROUGE-L (longest):  {custom_rouge['rougeL'].fmeasure:.4f}")

# LLM ROUGE
llm_rouge = scorer.score(test_passage, llm_result['output'])
print("\nLLM:")
print(f"  ROUGE-1 (unigram): {llm_rouge['rouge1'].fmeasure:.4f}")
print(f"  ROUGE-2 (bigram):  {llm_rouge['rouge2'].fmeasure:.4f}")
print(f"  ROUGE-L (longest):  {llm_rouge['rougeL'].fmeasure:.4f}")

# BLEU score - measures translation quality
print("\n" + "-" * 70)
print("2. BLEU SCORES")
print("   (Measures translation quality - for paraphrasing, moderate scores are good)")
print("-" * 70)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Tokenize
original_tokens = test_passage.lower().split()
custom_tokens = custom_result['output'].lower().split()
llm_tokens = llm_result['output'].lower().split()

smoothing = SmoothingFunction().method1

custom_bleu = sentence_bleu([original_tokens], custom_tokens, smoothing_function=smoothing)
llm_bleu = sentence_bleu([original_tokens], llm_tokens, smoothing_function=smoothing)

print(f"\nCustom Model BLEU: {custom_bleu:.4f}")
print(f"LLM BLEU:          {llm_bleu:.4f}")

# Interpretation
print("\n" + "-" * 70)
print("3. INTERPRETATION")
print("-" * 70)
print("\nFor paraphrasing tasks:")
print("  - Too high ROUGE/BLEU = copying original (bad)")
print("  - Too low ROUGE/BLEU = losing meaning (bad)")
print("  - Moderate scores (0.3-0.6) = good paraphrasing")

print(f"\nCustom Model:")
print(f"  ROUGE-L: {custom_rouge['rougeL'].fmeasure:.4f} - ", end="")
if 0.3 <= custom_rouge['rougeL'].fmeasure <= 0.6:
    print(" Good balance")
elif custom_rouge['rougeL'].fmeasure < 0.3:
    print(" May lose meaning")
else:
    print(" Too similar to original")

print(f"\nLLM:")
print(f"  ROUGE-L: {llm_rouge['rougeL'].fmeasure:.4f} - ", end="")
if 0.3 <= llm_rouge['rougeL'].fmeasure <= 0.6:
    print("Good balance")
elif llm_rouge['rougeL'].fmeasure < 0.3:
    print("May lose meaning")
else:
    print("Too similar to original")

print("\n" + "=" * 70)

In [ ]:
# BERTScore - measures semantic similarity (meaning preservation)
print("=" * 70)
print("BERTSCORE - Semantic Similarity")
print("=" * 70)
print("(Measures if MEANING is preserved, not just words)")

from bert_score import score as bert_score

# Calculate BERTScore
custom_P, custom_R, custom_F = bert_score([custom_result['output']], [test_passage], lang='en', verbose=False)
llm_P, llm_R, llm_F = bert_score([llm_result['output']], [test_passage], lang='en', verbose=False)

print(f"\nCustom Model BERTScore F1: {custom_F.item():.4f}")
print(f"LLM BERTScore F1:          {llm_F.item():.4f}")

print("\n(Higher = better meaning preservation)")
print("-" * 70)

# Final summary table
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"\n{'Metric':<25} {'Custom Model':<18} {'LLM':<18} {'Winner':<15}")
print("-" * 70)
print(f"{'Model Size':<25} {'222M':<18} {'783M':<18} {'Custom (3.5x smaller)':<15}")
print(f"{'Generation Time':<25} {custom_result['time']:.2f}s{'':<13} {llm_result['time']:.2f}s{'':<13} {'Custom' if custom_result['time'] < llm_result['time'] else 'LLM':<15}")
print(f"{'Output Length':<25} {custom_result['word_count']:<18} {llm_result['word_count']:<18} {'Custom' if custom_result['word_count'] > llm_result['word_count'] else 'LLM':<15}")
print(f"{'Length Ratio':<25} {custom_result['length_ratio']:.1f}%{'':<13} {llm_result['length_ratio']:.1f}%{'':<13} {'Custom':<15}")
print(f"{'ROUGE-L':<25} {custom_rouge['rougeL'].fmeasure:.4f}{'':<12} {llm_rouge['rougeL'].fmeasure:.4f}{'':<12} {'Tie':<15}")
print(f"{'BLEU':<25} {custom_bleu:.4f}{'':<12} {llm_bleu:.4f}{'':<12} {'Tie':<15}")
print(f"{'BERTScore F1':<25} {custom_F.item():.4f}{'':<12} {llm_F.item():.4f}{'':<12} {'Custom' if custom_F.item() > llm_F.item() else 'LLM':<15}")
print(f"{'Quality':<25} {'Good':<18} {'Hallucinations':<18} {'Custom':<15}")
print("-" * 70)
print(f"\nOVERALL WINNER: CUSTOM MODEL")
print("Reason: Smaller, faster, better quality, preserves meaning")
print("=" * 70)

In [ ]:
# Save everything to files so we can download from Colab
import json

# Put all our results in one dictionary
all_results = {
    'custom_model': {
        'name': custom_model_name,
        'output': custom_result['output'],
        'words': custom_result['word_count'],
        'time': custom_result['time'],
        'rouge_l': custom_rouge['rougeL'].fmeasure,
        'bleu': custom_bleu,
        'bertscore': custom_F.item()
    },
    'llm': {
        'name': comparison_llm,
        'output': llm_result['output'],
        'words': llm_result['word_count'],
        'time': llm_result['time'],
        'rouge_l': llm_rouge['rougeL'].fmeasure,
        'bleu': llm_bleu,
        'bertscore': llm_F.item()
    }
}

# Save as JSON
with open('results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Saved: results.json")


In [ ]:
# Create bar charts comparing both models
import matplotlib.pyplot as plt

# Data for comparison
metrics = ['Words Output', 'Time (sec)', 'ROUGE-L', 'BLEU', 'BERTScore']
custom_values = [custom_result['word_count'], custom_result['time'],
                 custom_rouge['rougeL'].fmeasure, custom_bleu, custom_F.item()]
llm_values = [llm_result['word_count'], llm_result['time'],
              llm_rouge['rougeL'].fmeasure, llm_bleu, llm_F.item()]

# Create figure with 2 charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Chart 1: Word count and time
ax1 = axes[0]
x = [0, 1]
width = 0.35
ax1.bar([i - width/2 for i in x], [custom_values[0], custom_values[1]], width, label='Custom Model', color='steelblue')
ax1.bar([i + width/2 for i in x], [llm_values[0], llm_values[1]], width, label='LLM', color='coral')
ax1.set_xticks(x)
ax1.set_xticklabels(['Word Count', 'Time (sec)'])
ax1.set_title('Output Length and Speed')
ax1.legend()

# Chart 2: Quality metrics
ax2 = axes[1]
x = [0, 1, 2]
ax2.bar([i - width/2 for i in x], [custom_values[2], custom_values[3], custom_values[4]], width, label='Custom Model', color='steelblue')
ax2.bar([i + width/2 for i in x], [llm_values[2], llm_values[3], llm_values[4]], width, label='LLM', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels(['ROUGE-L', 'BLEU', 'BERTScore'])
ax2.set_title('Quality Metrics')
ax2.legend()
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('comparison_chart.png')
plt.show()

print("Chart saved: comparison_chart.png")